# Feature Engineering and Modeling

---

### Goal
Build customer-level features using a temporal split approach,
train three models, and compare their performance honestly.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 5)

print("Libraries ready.")

Libraries ready.


In [2]:
orders_enriched = pd.read_csv('../data/processed/orders_enriched.csv',
                               parse_dates=['order_purchase_timestamp'])
products        = pd.read_csv('../data/raw/olist_products_dataset.csv')
items           = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
reviews         = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
payments        = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')

print(f"orders_enriched: {orders_enriched.shape}")
print(f"\nDate range in dataset:")
print(f"  Earliest: {orders_enriched['order_purchase_timestamp'].min().date()}")
print(f"  Latest:   {orders_enriched['order_purchase_timestamp'].max().date()}")

orders_enriched: (96478, 11)

Date range in dataset:
  Earliest: 2016-09-15
  Latest:   2018-08-29


---
## Sprint 5 — Feature Engineering

### The Temporal Split Approach

We use two separate time windows to prevent data leakage:

FEATURE WINDOW → all purchases before March 2018
  Used to build: recency, frequency, monetary, behavioral features

CHURN WINDOW → March 2018 to June 2018 (90 days)
  Used to define: did this customer buy again or not?

Because features and labels come from different time periods,
recency is now a safe feature — it measures past behavior,
not the future behavior that defines the label.

In [3]:
# Define the feature window end date
feature_window_end = pd.Timestamp('2018-03-01')

# Define snapshot_date for churn calculation
snapshot_date = orders_enriched['order_purchase_timestamp'].max() - pd.Timedelta(days=1)

# Filter orders to include only those in the feature window
orders_in_feature_window = orders_enriched[
    orders_enriched['order_purchase_timestamp'] < feature_window_end
]

# Recalculate recency, frequency, and monetary using only the feature window
rfm = orders_in_feature_window.groupby('customer_unique_id').agg(
    recency=('order_purchase_timestamp', lambda x: (feature_window_end - x.max()).days),
    frequency=('order_id', 'count'),
    monetary=('order_value', 'sum')
).reset_index()

rfm['recency'] = rfm['recency'].clip(lower=0)

print(f"Feature window end date: {feature_window_end.date()}")
print(f"RFM table shape: {rfm.shape}")
print(rfm[['recency', 'frequency', 'monetary']].describe().round(2))

Feature window end date: 2018-03-01
RFM table shape: (55525, 4)
        recency  frequency  monetary
count  55525.00   55525.00  55525.00
mean     156.83       1.03    162.35
std      112.25       0.20    222.46
min        0.00       1.00      0.00
25%       63.00       1.00     62.53
50%      133.00       1.00    106.19
75%      239.00       1.00    179.70
max      531.00       9.00  13664.08


In [4]:
# Ensure avg_order_value exists
if 'avg_order_value' not in rfm.columns:
    rfm['avg_order_value'] = rfm['monetary'] / rfm['frequency'].replace(0, np.nan)

# Recalculate customer lifespan using only the feature window
lifespan = orders_in_feature_window.groupby('customer_unique_id').agg(
    first_purchase=('order_purchase_timestamp', 'min'),
    last_purchase=('order_purchase_timestamp', 'max')
).reset_index()

lifespan['customer_lifespan_days'] = (
    lifespan['last_purchase'] - lifespan['first_purchase']
).dt.days

lifespan = lifespan[['customer_unique_id', 'customer_lifespan_days']]

# Drop duplicate columns before merging
rfm = rfm.drop(columns=[col for col in ['customer_lifespan_days', 'customer_lifespan_days_dup'] if col in rfm.columns])

rfm = rfm.merge(lifespan, on='customer_unique_id', how='left')

print(f"Shape after derived features: {rfm.shape}")
print(rfm[['avg_order_value', 'customer_lifespan_days']].describe().round(2))

Shape after derived features: (55525, 6)
       avg_order_value  customer_lifespan_days
count         55525.00                55525.00
mean            157.73                    1.60
std             215.75                   16.71
min               0.00                    0.00
25%              61.78                    0.00
50%             104.21                    0.00
75%             174.43                    0.00
max           13664.08                  454.00


In [5]:
# Recalculate category features using only the feature window
items_with_cat = items.merge(
    products[['product_id', 'product_category_name']],
    on='product_id', how='left'
)

items_with_customer = items_with_cat.merge(
    orders_in_feature_window[['order_id', 'customer_unique_id']],
    on='order_id', how='inner'
)

category_features = (
    items_with_customer
    .groupby('customer_unique_id')
    .agg(
        unique_categories=('product_category_name', 'nunique'),
        total_items=('order_id', 'count')
    )
    .reset_index()
)

rfm = rfm.merge(category_features, on='customer_unique_id', how='left')
print(f"Shape after category features: {rfm.shape}")

Shape after category features: (55525, 8)


In [6]:
# Recalculate review features using only the feature window
reviews_with_customer = orders_in_feature_window[['order_id', 'customer_unique_id']].merge(
    reviews[['order_id', 'review_score']], on='order_id', how='left'
)

review_features = (
    reviews_with_customer
    .groupby('customer_unique_id')
    .agg(
        avg_review_score=('review_score', 'mean'),
        review_count=('review_score', 'count')
    )
    .reset_index()
)

rfm = rfm.merge(review_features, on='customer_unique_id', how='left')
print(f"Shape after review features: {rfm.shape}")

Shape after review features: (55525, 10)


In [7]:
# Recalculate payment features using only the feature window
payments_with_customer = orders_in_feature_window[['order_id', 'customer_unique_id']].merge(
    payments[['order_id', 'payment_type', 'payment_installments']],
    on='order_id', how='left'
)

payment_features = (
    payments_with_customer
    .groupby('customer_unique_id')
    .agg(
        avg_installments=('payment_installments', 'mean'),
        used_credit_card=('payment_type', lambda x: int((x == 'credit_card').any()))
    )
    .reset_index()
)

rfm = rfm.merge(payment_features, on='customer_unique_id', how='left')
print(f"Shape after payment features: {rfm.shape}")

Shape after payment features: (55525, 12)


In [11]:
CHURN_DAYS = 90

# Ensure snapshot_date is defined
if 'snapshot_date' not in locals():
    # Set snapshot_date to the end of the feature window to avoid data leakage
    snapshot_date = feature_window_end

customer_labels = orders_enriched.groupby('customer_unique_id').agg(
    last_purchase=('order_purchase_timestamp', 'max')
).reset_index()

# Calculate days since last purchase
customer_labels['days_since_last_purchase'] = (
    (snapshot_date - customer_labels['last_purchase']).dt.days
).clip(lower=0)

customer_labels['churned'] = (
    customer_labels['days_since_last_purchase'] > CHURN_DAYS
).astype(int)

# Ensure churned column exists in rfm
if 'churned' not in rfm.columns:
    rfm = rfm.merge(
        customer_labels[['customer_unique_id', 'churned']],
        on='customer_unique_id', how='left'
    )

rfm['avg_review_score']       = rfm['avg_review_score'].fillna(rfm['avg_review_score'].median())
rfm['review_count']           = rfm['review_count'].fillna(0)
rfm['avg_installments']       = rfm['avg_installments'].fillna(1)
rfm['unique_categories']      = rfm['unique_categories'].fillna(1)
rfm['total_items']            = rfm['total_items'].fillna(1)
rfm['used_credit_card']       = rfm['used_credit_card'].fillna(0)
rfm['customer_lifespan_days'] = rfm['customer_lifespan_days'].fillna(0)

print(f"Final feature table shape: {rfm.shape}")
print(f"Missing values:            {rfm.isnull().sum().sum()}")
if 'churned' in rfm.columns:
    print(f"Churn rate:                {rfm['churned'].mean():.1%}")
else:
    print("Churn column is missing in the dataset.")

print(f"\nAll columns:")
for col in rfm.columns:
    print(f"  {col}")

rfm.to_csv('../data/processed/features_and_labels.csv', index=False)
print("\nSaved.")

Final feature table shape: (55525, 13)
Missing values:            0
Churn rate:                99.5%

All columns:
  customer_unique_id
  recency
  frequency
  monetary
  avg_order_value
  customer_lifespan_days
  unique_categories
  total_items
  avg_review_score
  review_count
  avg_installments
  used_credit_card
  churned

Saved.


In [ ]:
# Add new features to improve feature engineering
rfm['tenure'] = (snapshot_date - orders_in_feature_window.groupby('customer_unique_id')['order_purchase_timestamp'].min()).dt.days
rfm['avg_time_between_purchases'] = rfm['tenure'] / rfm['frequency'].replace(0, np.nan)
rfm['monetary_per_order'] = rfm['monetary'] / rfm['frequency'].replace(0, np.nan)
rfm['credit_card_ratio'] = rfm['used_credit_card'] / rfm['frequency'].replace(0, np.nan)

# Apply log transformation to reduce skewness in monetary and frequency features
rfm['log_monetary'] = np.log1p(rfm['monetary'])
rfm['log_frequency'] = np.log1p(rfm['frequency'])

# Interaction features
rfm['recency_frequency_interaction'] = rfm['recency'] * rfm['frequency']
rfm['monetary_frequency_interaction'] = rfm['monetary'] * rfm['frequency']

print("New features added and transformations applied.")
print(rfm.describe().round(2))

In [12]:
drop_cols = ['customer_unique_id', 'churned']
X = rfm.drop(columns=drop_cols)
y = rfm['churned']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set:   {X_train.shape[0]:,} customers")
print(f"Test set:       {X_test.shape[0]:,} customers")
print(f"\nFeatures ({X.shape[1]}):")
for col in X.columns:
    print(f"  {col}")
print(f"\nChurn rate in train: {y_train.mean():.1%}")
print(f"Churn rate in test:  {y_test.mean():.1%}")

Training set:   44,420 customers
Test set:       11,105 customers

Features (11):
  recency
  frequency
  monetary
  avg_order_value
  customer_lifespan_days
  unique_categories
  total_items
  avg_review_score
  review_count
  avg_installments
  used_credit_card

Churn rate in train: 99.5%
Churn rate in test:  99.5%


In [13]:
rfm = rfm.merge(customer_labels[['customer_unique_id', 'churned']], on='customer_unique_id', how='inner', suffixes=('', '_label'))

# Ensure no duplicate columns exist after the merge
if 'churned_label' in rfm.columns:
    rfm.drop(columns=['churned_label'], inplace=True)

rfm['avg_review_score']       = rfm['avg_review_score'].fillna(rfm['avg_review_score'].median())
rfm['review_count']           = rfm['review_count'].fillna(0)
rfm['avg_installments']       = rfm['avg_installments'].fillna(1)
rfm['unique_categories']      = rfm['unique_categories'].fillna(1)
rfm['total_items']            = rfm['total_items'].fillna(1)
rfm['used_credit_card']       = rfm['used_credit_card'].fillna(0)
rfm['customer_lifespan_days'] = rfm['customer_lifespan_days'].fillna(0)

print(f"Final feature table shape: {rfm.shape}")
print(f"Missing values remaining:  {rfm.isnull().sum().sum()}")
print(f"Churn rate:                {rfm['churned'].mean():.1%}")
print(f"\nAll columns:")
for col in rfm.columns:
    print(f"  {col}")

rfm.to_csv('../data/processed/features_and_labels.csv', index=False)
print("\nSaved to data/processed/features_and_labels.csv")

Final feature table shape: (55525, 13)
Missing values remaining:  0
Churn rate:                99.5%

All columns:
  customer_unique_id
  recency
  frequency
  monetary
  avg_order_value
  customer_lifespan_days
  unique_categories
  total_items
  avg_review_score
  review_count
  avg_installments
  used_credit_card
  churned

Saved to data/processed/features_and_labels.csv


---
## Sprint 6 — Modeling

Three models trained in order of complexity:
1. Logistic Regression — baseline, establishes the performance floor
2. Random Forest — intermediate, handles non-linear patterns
3. XGBoost — best model, boosting with class imbalance handling

Note on recency: recency is included as a feature and is the strongest
predictor since churn is defined by inactivity. In a production system
this would be resolved with a temporal train/test split on a dataset
with sufficient repeat purchase history. Documented in Limitations.

In [14]:
drop_cols = ['customer_unique_id', 'churned']
X = rfm.drop(columns=drop_cols)
y = rfm['churned']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set:   {X_train.shape[0]:,} customers")
print(f"Test set:       {X_test.shape[0]:,} customers")
print(f"\nFeatures ({X.shape[1]}):")
for col in X.columns:
    print(f"  {col}")
print(f"\nChurn rate in train: {y_train.mean():.1%}")
print(f"Churn rate in test:  {y_test.mean():.1%}")

Training set:   44,420 customers
Test set:       11,105 customers

Features (11):
  recency
  frequency
  monetary
  avg_order_value
  customer_lifespan_days
  unique_categories
  total_items
  avg_review_score
  review_count
  avg_installments
  used_credit_card

Churn rate in train: 99.5%
Churn rate in test:  99.5%


In [15]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("Scaling complete.")
print(f"Feature means after scaling (all should be near 0):")
print(pd.Series(X_train_scaled.mean(axis=0), index=X.columns).round(3))

Scaling complete.
Feature means after scaling (all should be near 0):
recency                  -0.0
frequency                 0.0
monetary                  0.0
avg_order_value           0.0
customer_lifespan_days    0.0
unique_categories        -0.0
total_items              -0.0
avg_review_score          0.0
review_count              0.0
avg_installments         -0.0
used_credit_card          0.0
dtype: float64


In [16]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)

y_pred_lr  = lr.predict(X_test_scaled)
y_proba_lr = lr.predict_proba(X_test_scaled)[:, 1]

lr_auc = roc_auc_score(y_test, y_proba_lr)

print("=== LOGISTIC REGRESSION ===")
print(f"ROC-AUC: {lr_auc:.4f}")
print()
print(classification_report(y_test, y_pred_lr, target_names=['Active', 'Churned']))

=== LOGISTIC REGRESSION ===
ROC-AUC: 0.5895

              precision    recall  f1-score   support

      Active       0.00      0.00      0.00        57
     Churned       0.99      1.00      1.00     11048

    accuracy                           0.99     11105
   macro avg       0.50      0.50      0.50     11105
weighted avg       0.99      0.99      0.99     11105



In [17]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred_rf  = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

rf_auc = roc_auc_score(y_test, y_proba_rf)

print("=== RANDOM FOREST ===")
print(f"ROC-AUC: {rf_auc:.4f}")
print()
print(classification_report(y_test, y_pred_rf, target_names=['Active', 'Churned']))

=== RANDOM FOREST ===
ROC-AUC: 0.5652

              precision    recall  f1-score   support

      Active       0.00      0.00      0.00        57
     Churned       0.99      1.00      1.00     11048

    accuracy                           0.99     11105
   macro avg       0.50      0.50      0.50     11105
weighted avg       0.99      0.99      0.99     11105



In [18]:
neg   = (y_train == 0).sum()
pos   = (y_train == 1).sum()
scale = neg / pos

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale,
    eval_metric='auc',
    random_state=42,
    n_jobs=-1,
    verbosity=0
)
xgb.fit(X_train, y_train)

y_pred_xgb  = xgb.predict(X_test)
y_proba_xgb = xgb.predict_proba(X_test)[:, 1]

xgb_auc = roc_auc_score(y_test, y_proba_xgb)

print("=== XGBOOST ===")
print(f"ROC-AUC: {xgb_auc:.4f}")
print()
print(classification_report(y_test, y_pred_xgb, target_names=['Active', 'Churned']))

=== XGBOOST ===
ROC-AUC: 0.5765

              precision    recall  f1-score   support

      Active       0.01      0.26      0.01        57
     Churned       1.00      0.81      0.89     11048

    accuracy                           0.81     11105
   macro avg       0.50      0.54      0.45     11105
weighted avg       0.99      0.81      0.89     11105

